In [1]:
import pandas as pd
import numpy as np
import pathlib
from tqdm.auto import tqdm

import hydra
from omegaconf import DictConfig, OmegaConf

import torch
from torch_geometric import seed_everything

import ray

In [2]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

In [3]:
records = pd.read_feather(f"{output_path}/baseline_records_220412.feather").set_index("eid")

In [4]:
records.info()

In [5]:
records_freq = records.sum(axis=0).sort_values(ascending=False).to_frame().reset_index()
records_freq.columns = ["record", "n"]
records_freq = records_freq.set_index("record")
records_freq

                   n
record              
OMOP_4081598  309295
OMOP_4052351  271672
OMOP_4061103  264875
OMOP_4144272  247882
OMOP_4057411  222759
...              ...
OMOP_4125272       0
OMOP_4236239       0
OMOP_4236199       0
OMOP_4236188       0
OMOP_4107275       0

[68527 rows x 1 columns]

In [6]:
concepts_raw = pd.read_csv("/sc-projects/sc-proj-ukb-cvd/data/mapping/athena/CONCEPT.csv", sep="\t", engine="c", dtype={"concept_id": str})
concepts_raw["record"] = "OMOP_" + concepts_raw["concept_id"]
concept_raw = concepts_raw.set_index("record")

In [7]:
records_freq_md = records_freq.merge(concept_raw, left_index=True, right_index=True, how="left")

In [8]:
records_freq_md.query("n>10000")

                    n concept_id                  concept_name    domain_id  \
record                                                                        
OMOP_4081598   309295    4081598     Notes summary on computer  Observation   
OMOP_4052351   271672    4052351                Alcohol intake  Observation   
OMOP_4061103   264875    4061103  O/E - blood pressure reading    Condition   
OMOP_4144272   247882    4144272          Never smoked tobacco  Observation   
OMOP_4057411   222759    4057411          Review of medication    Procedure   
...               ...        ...                           ...          ...   
OMOP_1189754    10068    1189754                     rofecoxib         Drug   
OMOP_4197456    10057    4197456            Colonoscopy normal    Condition   
OMOP_193871     10056     193871   Genuine stress incontinence    Condition   
OMOP_4150400    10041    4150400                  Diet average  Observation   
OMOP_19040060   10025   19040060                dydr

In [9]:
records_freq_md.query("n>100")

                    n concept_id  \
record                             
OMOP_4081598   309295    4081598   
OMOP_4052351   271672    4052351   
OMOP_4061103   264875    4061103   
OMOP_4144272   247882    4144272   
OMOP_4057411   222759    4057411   
...               ...        ...   
OMOP_44791099     101   44791099   
OMOP_4076060      101    4076060   
OMOP_4165739      101    4165739   
OMOP_4206307      101    4206307   
OMOP_4038926      101    4038926   

                                                    concept_name    domain_id  \
record                                                                          
OMOP_4081598                           Notes summary on computer  Observation   
OMOP_4052351                                      Alcohol intake  Observation   
OMOP_4061103                        O/E - blood pressure reading    Condition   
OMOP_4144272                                Never smoked tobacco  Observation   
OMOP_4057411                                R

In [10]:
records_freq_md.query("n>1000")

                    n concept_id  \
record                             
OMOP_4081598   309295    4081598   
OMOP_4052351   271672    4052351   
OMOP_4061103   264875    4061103   
OMOP_4144272   247882    4144272   
OMOP_4057411   222759    4057411   
...               ...        ...   
OMOP_45772085    1001   45772085   
OMOP_44790912    1001   44790912   
OMOP_434926      1001     434926   
OMOP_19055344    1001   19055344   
OMOP_4203473     1001    4203473   

                                                    concept_name    domain_id  \
record                                                                          
OMOP_4081598                           Notes summary on computer  Observation   
OMOP_4052351                                      Alcohol intake  Observation   
OMOP_4061103                        O/E - blood pressure reading    Condition   
OMOP_4144272                                Never smoked tobacco  Observation   
OMOP_4057411                                R

In [11]:
records_freq_md.query("n>100")

                    n concept_id  \
record                             
OMOP_4081598   309295    4081598   
OMOP_4052351   271672    4052351   
OMOP_4061103   264875    4061103   
OMOP_4144272   247882    4144272   
OMOP_4057411   222759    4057411   
...               ...        ...   
OMOP_44791099     101   44791099   
OMOP_4076060      101    4076060   
OMOP_4165739      101    4165739   
OMOP_4206307      101    4206307   
OMOP_4038926      101    4038926   

                                                    concept_name    domain_id  \
record                                                                          
OMOP_4081598                           Notes summary on computer  Observation   
OMOP_4052351                                      Alcohol intake  Observation   
OMOP_4061103                        O/E - blood pressure reading    Condition   
OMOP_4144272                                Never smoked tobacco  Observation   
OMOP_4057411                                R

In [12]:
records_freq_md.query("n>50")

                   n concept_id                     concept_name    domain_id  \
record                                                                          
OMOP_4081598  309295    4081598        Notes summary on computer  Observation   
OMOP_4052351  271672    4052351                   Alcohol intake  Observation   
OMOP_4061103  264875    4061103     O/E - blood pressure reading    Condition   
OMOP_4144272  247882    4144272             Never smoked tobacco  Observation   
OMOP_4057411  222759    4057411             Review of medication    Procedure   
...              ...        ...                              ...          ...   
OMOP_4011594      51    4011594                   Microbiologist  Observation   
OMOP_4050667      51    4050667        Road transport occupation  Observation   
OMOP_4228802      51    4228802  Mild recurrent major depression    Condition   
OMOP_4106050      51    4106050      Arthroscopic knee operation    Procedure   
OMOP_381020       51     381

In [13]:
records_freq_md.query("n>25")

                   n concept_id                                concept_name  \
record                                                                        
OMOP_4081598  309295    4081598                   Notes summary on computer   
OMOP_4052351  271672    4052351                              Alcohol intake   
OMOP_4061103  264875    4061103                O/E - blood pressure reading   
OMOP_4144272  247882    4144272                        Never smoked tobacco   
OMOP_4057411  222759    4057411                        Review of medication   
...              ...        ...                                         ...   
OMOP_4023106      26    4023106  Cystoscopic dilatation of ureteric orifice   
OMOP_4220185      26    4220185                           Craniopharyngioma   
OMOP_4176894      26    4176894                                 Liposarcoma   
OMOP_4090859      26    4090859         Well adult monitoring second letter   
OMOP_4157449      26    4157449            Malignant

In [14]:
records_freq_md.query("n>25").shape

(19122, 11)

In [15]:
records_freq_md.query("n>50").shape

(15190, 11)

In [16]:
records_freq_md.query("n>100").shape

(11762, 11)

In [17]:
records_freq_md.query("n>1000").shape

(3711, 11)

In [18]:
records_freq_md.query("n>10").shape

(25132, 11)

In [19]:
records_freq_md.query("n>5").shape

(30023, 11)

In [20]:
artifact_path = "/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/record_frequencies.feather"

In [21]:
records_freq_md.reset_index().to_feather(artifact_path)

In [22]:
records_freq_md

                   n concept_id                               concept_name  \
record                                                                       
OMOP_4081598  309295    4081598                  Notes summary on computer   
OMOP_4052351  271672    4052351                             Alcohol intake   
OMOP_4061103  264875    4061103               O/E - blood pressure reading   
OMOP_4144272  247882    4144272                       Never smoked tobacco   
OMOP_4057411  222759    4057411                       Review of medication   
...              ...        ...                                        ...   
OMOP_4125272       0    4125272                            Able to swallow   
OMOP_4236239       0    4236239  Functional defects of methionine synthase   
OMOP_4236199       0    4236199                   Tends not to be sociable   
OMOP_4236188       0    4236188                    Coronavirus vaccination   
OMOP_4107275       0    4107275      Revision arthroscopic ligam

In [23]:
artifact_path = "/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/record_frequencies_220512.feather"

In [24]:
records_freq_md.reset_index().to_feather(artifact_path)

In [25]:
records_freq_md.reset_index()[["record", "n", "concept_id", "concept_name", "domain_id", "vocabulary_id", "concept_class_id", "standard_concept"]].to_feather(artifact_path)

In [26]:
import wandb

run = wandb.init(project="RecordGraphs", entity="cardiors", tags=["artifacts"])

artifact = wandb.Artifact("RecordFrequencies", type="prepare_records")
artifact.add_reference(f"file://{artifact_path}", "RecordsMetadata", checksum=True)
run.log_artifact(artifact)

run.finish()